# PINTHAC Quick-Start Tutorial

This notebook is a brief introduction to some of the core capabilities of **PINTHAC**. It covers:

1. **Water property lookups** using the IAPWS-95 formulation (single points, arrays, and GPU tensors)
2. **Assembling property dictionaries** with the `getprop` function for multiple fluids
3. **Computing correlations** for heat transfer coefficients, including supercritical water

Throughout, results are benchmarked against values computed in EES (Engineering Equation Solver).

> **Note:** The benchmark files `density.txt` and `htc_ees.txt` should be placed in the same folder as this notebook.

## Setup

Import the libraries used throughout the tutorial and set the path to the benchmark data.

In [ ]:
import os

import numpy as np
import matplotlib.pyplot as plt
import torch

import pinthac

# Directory containing the EES benchmark files (defaults to the notebook's folder)
path = os.getcwd()

---
## 1. General Water Property Lookups

Water properties are computed with the IAPWS-95 formulation, available in the `pinthac.properties.iapws95` module.

In [ ]:
import pinthac.properties.iapws95 as iapws

### 1.1 Computing density from temperature and pressure

At a given temperature and pressure, first compute the water density using `rho_Tp`.

**Units:** temperature is in **kelvin** and pressure is in **MPa**.

In [ ]:
Tin = 385 + 273.15   # Temperature [K]
Pin = 25             # Pressure [MPa]

rho_test = iapws.IAPWS95.rho_Tp(Tin, Pin)
print(f"rho_test is {rho_test:.2f} kg/m3")

### 1.2 Computing the thermodynamic state

With the density known, compute the **state** variable from the Helmholtz free energy formulation. This state is the input used to evaluate every other property.

In [ ]:
state = iapws.IAPWS95.helmholtz(rho_test, Tin)

### 1.3 Computing individual properties

Individual properties can now be evaluated from the state. Many functions accept a `units` argument; for example, specific heat can be returned in J or kJ.

In [ ]:
cp = iapws.IAPWS95.cp(state, units='J')
cp_kJ = iapws.IAPWS95.cp(state, units='kJ')

print(f"The computed specific heat in J/kg-K is {cp:.2f}")
print(f"The computed specific heat in kJ/kg-K is {cp_kJ:.2f}")

### 1.4 Batch calculations with arrays

The property functions also accept arrays, so a whole batch of state points can be computed at once. Here density is computed over a temperature sweep through the pseudo-critical region at 25 MPa. Scalar inputs (like the pressure) are broadcast automatically.

The `newton_iters` argument raises the iteration limit of the density solver, which helps convergence near the pseudo-critical point.

In [ ]:
Tmin, Tmax = 350 + 273.15, 415 + 273.15   # Temperature range [K]
n = 50                                     # Number of points

Tvec = np.linspace(Tmin, Tmax, n)
pvec = 25   # Pressure [MPa]; a scalar is broadcast across Tvec

rhovec = iapws.IAPWS95.rho_Tp(Tvec, pvec, newton_iters=120)
print("rhovec is", np.round(rhovec, 2))

### 1.5 Benchmarking against EES

Load the EES density values and compare them with the library results.

In [ ]:
ees_dat = np.loadtxt(os.path.join(path, 'density.txt'))
rho_ees = ees_dat[:, 1]

err_rel = np.abs(rhovec - rho_ees) / rho_ees

In [ ]:
plt.figure()
plt.plot(Tvec, rho_ees, color='blue', label='EES values', linewidth=1.8)
plt.plot(Tvec, rhovec, color='orange', label='Library values', linestyle='--', linewidth=1.8)
plt.xlabel('Temperature [K]')
plt.ylabel('Density [kg/m$^3$]')
plt.title('Density at 25 MPa: PINTHAC vs. EES')
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(Tvec, 100 * err_rel, color='black', linewidth=1.8)
plt.xlabel('Temperature [K]')
plt.ylabel('Relative Error [%]')
plt.title('Density Relative Error vs. EES')
plt.show()

### 1.6 Running on the GPU with PyTorch

PINTHAC also accepts PyTorch tensors, which lets calculations run on a GPU if one is available. Testing shows the GPU becomes faster than the CPU at roughly **100+ state points**.

First, select the device and move a tensor onto it:

In [ ]:
# Use the GPU if available, otherwise fall back to the CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Build the temperature tensor and move it to the device
Tvec_torch = torch.linspace(Tmin, Tmax, n)
Tvec_torch_gpu = Tvec_torch.to(device)

After performing calculations with tensors on the GPU, bring the results back to the CPU and convert them to NumPy before plotting or doing further NumPy work:

In [ ]:
Tvec_np_cpu = Tvec_torch_gpu.detach().cpu().numpy()

---
## 2. Property Dictionaries with `getprop`

The correlations module takes a **`Prop`** argument: a dictionary of the key fluid and transport properties. Each value has the same type as the input temperature.

| Key | Property | Available for |
|---|---|---|
| `rho` | Density | All substances |
| `h` | Enthalpy | All substances |
| `cp` | Specific heat | All substances |
| `mu` | Dynamic viscosity | All substances |
| `k` | Thermal conductivity | All substances |
| `sigma` | Surface tension | `"Water"` and all liquid metals |

The keys do not depend on the chosen `formulation`.

The `getprop` function assembles this dictionary quickly for many materials. Liquid metals don't require a pressure, so `P=None` is passed.

In [ ]:
import pinthac.properties.getprop as gp

Tval = 380 + 273.15   # Temperature [K]
pval = 27             # Pressure [MPa]

prop_SCW = gp._getprop('SCW', Tval, pval)        # Supercritical water
prop_Lead = gp._getprop('Lead', Tval, P=None)    # Liquid lead
prop_Sodium = gp._getprop('Sodium', Tval, P=None)  # Liquid sodium

---
## 3. Correlations

The `pinthac.correlations` package provides **friction factors**, **heat transfer coefficients (HTC)**, and **rod bundle correction factors**. Each module contains a class per material, and each class contains its available correlations.

> **Tip:** In VS Code (or Jupyter with autocomplete), typing `htc.Water.` will show a preview of all HTC correlations available for water.

### 3.1 Basic HTC correlations

Compute the HTC for supercritical water (Dittus–Boelter) and liquid sodium (Lyon) given a mass flux and hydraulic diameter.

In [ ]:
import pinthac.correlations.htc as htc
import pinthac.correlations.friction as fric

G = 800     # Mass flux [kg/m2-s]
D = 0.007   # Hydraulic diameter [m]

htc_dittus = htc.Water.Dittus(prop_SCW, G, D)
htc_Sodium = htc.Sodium.Lyon(prop_Sodium, G, D)

print(f"HTC (Dittus-Boelter, SCW) is {htc_dittus:.2f} W/m2-K")
print(f"HTC (Lyon, sodium) is {htc_Sodium:.2f} W/m2-K")

### 3.2 Supercritical water: the Swenson correlation

Some supercritical water correlations, such as **Swenson**, depend on properties evaluated at the wall temperature, which is itself unknown. For these, you must also supply:

- the **heat flux** `qpp` [W/m²]
- a **function** that returns the property dictionary at a given wall temperature

The library then iterates to find the consistent wall temperature. The `hi` argument sets the upper bound of the wall-temperature search.

First, load the EES benchmark data:

In [ ]:
eesvals = np.loadtxt(os.path.join(path, 'htc_ees.txt'))

qpp        = eesvals[:, 0]   # Heat flux [W/m2]
Tw_sw_ees  = eesvals[:, 2]   # Swenson wall temperature from EES
htc_db_ees = eesvals[:, 3]   # Dittus-Boelter HTC from EES
htc_sw_ees = eesvals[:, 4]   # Swenson HTC from EES

Next, set up the bulk properties and the wall-property function, then evaluate the correlation:

In [ ]:
Tbulk = 380 + 273.15            # Bulk temperature [K]
pval = 27 * np.ones_like(qpp)   # Pressure [MPa], one value per heat flux

prop_bulk = gp._getprop('SCW', Tbulk, pval)

def prop_wall(Twall):
    """Return the SCW property dictionary evaluated at the wall temperature."""
    return gp._getprop('SCW', Twall, pval)

htc_sw = htc.SCW.Swenson(prop_bulk, prop_wall, G, D, qpp, Tbulk, hi=Tbulk + 1000)

# Wall temperature from Newton's law of cooling, converted to degrees C
Tw_sw = Tbulk + qpp / htc_sw - 273.15

### 3.3 Benchmarking Swenson against EES

In [ ]:
sw_err_rel = np.abs(htc_sw - htc_sw_ees) / htc_sw_ees
Tw_err_rel = np.abs(Tw_sw - Tw_sw_ees) / Tw_sw_ees

qppkW = qpp / 1e3   # Heat flux [kW/m2]

In [ ]:
plt.figure()
plt.plot(qppkW, htc_sw_ees, label='EES Swenson', color='blue', linewidth=1.8)
plt.plot(qppkW, htc_sw, label='Library Swenson', color='orange', linewidth=1.8, linestyle='--')
plt.xlabel('Heat Flux [kW/m$^2$]')
plt.ylabel('HTC [W/m$^2$-K]')
plt.title('Swenson HTC: PINTHAC vs. EES')
plt.legend()
plt.show()

In [ ]:
plt.figure()
plt.plot(qppkW, 100 * Tw_err_rel, color='black', linewidth=1.8)
plt.xlabel('Heat Flux [kW/m$^2$]')
plt.ylabel('Wall Temperature Relative Error [%]')
plt.title('Predicted Wall Temperature Error vs. EES')
plt.show()

---
## Summary

In this tutorial you learned how to:

- Compute water density and properties with the **IAPWS-95** routines, for single points or whole arrays
- Run property calculations on the **GPU** using PyTorch tensors
- Build property dictionaries for water and liquid metals with **`getprop`**
- Evaluate **HTC correlations**, including the wall-property-dependent **Swenson** correlation for supercritical water